# XGBoost Dark Matter Estimation on SPARC Data
1. Combines the 4 SPARC training datasets (`galaxies.csv`, `stellar-masses.csv`, `BTF.csv`, `newt-mass.csv`) into a single master file `data/sparc_combined_train.csv` while keeping all 4 separate CSVs intact.
2. Calculates physical target Dark Matter Fraction ($f_{\text{DM}}$) from rotation curve decomposition at outer radius $R_{\text{max}}$.
3. Preprocesses galaxy features (masses, radii, surface brightness, rotation speed, color, morph type).
4. Trains an XGBoost regression model with 5-fold cross-validation.
5. Uses SHAP to identify which observable properties most determine dark matter fraction and dark halo mass.


In [ ]:
# Run once — comment out after first run
# !pip install xgboost scikit-learn pandas numpy shap --quiet

In [ ]:
import os
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("All imports successful.")
print(f"XGBoost version: {xgb.__version__}")

In [ ]:
USE_SYNTHETIC      = False          # Set True only for synthetic test fallback
TRAIN_DIR          = "data/train"
COMBINED_CSV_PATH  = "data/sparc_combined_train.csv"

FEATURE_COLS = [
    "stellar_mass_logM",
    "HI_gas_mass_MHI",
    "baryonic_mass_logMb",
    "effective_radius_Reff",
    "surface_brightness_SBeff",
    "disk_scale_length_Rdisk",
    "rotation_velocity_Vflat",
    "color_3p6_4p5",
    "morph_type_T",
]

TARGET_COL   = "dm_fraction"
RANDOM_STATE = 42
N_FOLDS      = 5

In [ ]:
from create_combined_sparc_dataset import build_combined_sparc_dataset

def make_synthetic_sparc(n=175, seed=42):
    """Synthetic SPARC-like fallback data."""
    rng = np.random.default_rng(seed)
    stellar_mass = 10 ** rng.uniform(7, 12, n) / 1e9
    gas_mass     = stellar_mass * rng.uniform(0.05, 2.0, n)
    luminosity   = stellar_mass * rng.uniform(0.5, 3.0, n)
    radius       = rng.uniform(1, 30, n)
    surf_bright  = luminosity / (np.pi * radius**2)
    vflat        = rng.uniform(50, 350, n)
    dm_fraction  = np.clip(
        1.0 - surf_bright / surf_bright.max() * 0.8 + rng.normal(0, 0.1, n),
        0.05, 0.98
    )
    return pd.DataFrame({
        "Galaxy":                      [f"SYN_{i:03d}" for i in range(n)],
        "stellar_mass_logM":           np.log10(stellar_mass * 1e9),
        "HI_gas_mass_MHI":             gas_mass,
        "baryonic_mass_logMb":         np.log10((stellar_mass + gas_mass) * 1e9),
        "effective_radius_Reff":       radius,
        "surface_brightness_SBeff":    surf_bright,
        "disk_scale_length_Rdisk":     radius * 0.6,
        "rotation_velocity_Vflat":     vflat,
        "color_3p6_4p5":               rng.uniform(2.5, 4.5, n),
        "morph_type_T":                rng.integers(1, 11, n),
        "dm_fraction":                 dm_fraction,
    })

if USE_SYNTHETIC:
    df = make_synthetic_sparc()
    print("Using synthetic data (175 galaxies)")
else:
    # Generate combined dataset from 4 separate CSVs if not present
    if not os.path.exists(COMBINED_CSV_PATH):
        df = build_combined_sparc_dataset(train_dir=TRAIN_DIR, output_csv=COMBINED_CSV_PATH)
    else:
        df = pd.read_csv(COMBINED_CSV_PATH)
        print(f"Loaded combined dataset from '{COMBINED_CSV_PATH}' ({len(df)} galaxies)")

print(f"Shape: {df.shape}")
df[["Galaxy", "stellar_mass_logM", "HI_gas_mass_MHI", "baryonic_mass_logMb", "effective_radius_Reff", "surface_brightness_SBeff", "rotation_velocity_Vflat", "dm_fraction"]].head()

In [ ]:
available = [c for c in FEATURE_COLS if c in df.columns]
missing   = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    print(f"WARNING — missing columns (skipped): {missing}")

df_clean = df[available + [TARGET_COL]].dropna().reset_index(drop=True)
print(f"Galaxies after dropping NaN rows: {len(df_clean)}")

X = df_clean[available].copy()
y = df_clean[TARGET_COL].values
feature_names = list(X.columns)
X_arr = X.values.astype(float)

print(f"Feature matrix shape: {X_arr.shape}")
print(f"DM fraction range: [{y.min():.3f}, {y.max():.3f}]")
X.describe()

### XGBoost Regressor Configuration
| Parameter | Value | Rationale |
|---|---|---|
| `max_depth` | 3 | Shallow trees prevent overfitting on small astronomical datasets |
| `learning_rate` | 0.05 | Shrinkage step size for gradual ensemble learning |
| `subsample` | 0.8 | Row subsampling per tree |
| `colsample_bytree` | 0.8 | Feature subsampling per tree |
| `reg_alpha` | 0.1 | L1 regularization |
| `reg_lambda` | 1.0 | L2 regularization |


In [ ]:
def build_model():
    return xgb.XGBRegressor(
        n_estimators     = 300,
        max_depth        = 3,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        reg_alpha        = 0.1,
        reg_lambda       = 1.0,
        objective        = "reg:squarederror",
        eval_metric      = "rmse",
        random_state     = RANDOM_STATE,
        verbosity        = 0,
    )

print("Model config:")
print(build_model())

In [ ]:
try:
    dm_bins = pd.qcut(y, q=N_FOLDS, labels=False, duplicates="drop")
    kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    splits = list(kf.split(X_arr, dm_bins))
except Exception:
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    splits = list(kf.split(X_arr))

fold_results = []
all_y_true   = []
all_y_pred   = []
all_shap     = []

for fold, (train_idx, test_idx) in enumerate(splits):
    X_train, X_test = X_arr[train_idx], X_arr[test_idx]
    y_train, y_test = y[train_idx],     y[test_idx]

    model = build_model()
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    y_pred = model.predict(X_test)
    r2     = r2_score(y_test, y_pred)
    mae    = mean_absolute_error(y_test, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

    fold_results.append({"Fold": fold+1, "R2": r2, "MAE": mae, "RMSE": rmse})
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)

    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X_test)
    all_shap.append(np.abs(shap_vals))

    print(f"Fold {fold+1}: R²={r2:+.3f}  MAE={mae:.4f}  RMSE={rmse:.4f}")

df_folds = pd.DataFrame(fold_results)
print(f"\nMean R²  : {df_folds['R2'].mean():.3f} ± {df_folds['R2'].std():.3f}")
print(f"Mean MAE : {df_folds['MAE'].mean():.4f} ± {df_folds['MAE'].std():.4f}")
print(f"Mean RMSE: {df_folds['RMSE'].mean():.4f} ± {df_folds['RMSE'].std():.4f}")

In [ ]:
# Fold performance summary table
df_folds.round({"R2": 3, "MAE": 4, "RMSE": 4})

In [ ]:
# Train final model on full dataset
final_model = build_model()
final_model.fit(X_arr, y, verbose=False)

explainer_final = shap.TreeExplainer(final_model)
shap_values     = explainer_final.shap_values(X_arr)

# Compute global feature importance
mean_abs_shap = np.abs(shap_values).mean(axis=0)

importance_df = pd.DataFrame({
    "Feature":        feature_names,
    "Mean |SHAP|":    mean_abs_shap,
}).sort_values("Mean |SHAP|", ascending=False).reset_index(drop=True)

importance_df["Rank"] = range(1, len(importance_df)+1)
print("Feature importance ranking (SHAP):")
importance_df[["Rank","Feature","Mean |SHAP|"]]

In [ ]:
# ANSWERS RESEARCH QUESTION: Which observable properties are most determinative of dark matter fraction
shap.summary_plot(
    shap_values,
    X_arr,
    feature_names = feature_names,
    plot_type     = "bar",
    show          = True,
)